# 06 — De los embeddings a la recuperación: RAG

**Módulo V — Diplomado de Ciencia de Datos (FES Acatlán, UNAM)**
**Sesión 10 · Bloque 3 de 3 · Duración estimada: 50 minutos**

**Notas de clase:** capítulo 2, subsección *"De embeddings a recuperación de información:
introducción a RAG"*.
**Prerrequisitos:** los notebooks `04_Intro_to_Embeddings.ipynb` y
`05 Token_vs_Sentence_Embeddings.ipynb` de este mismo bloque.

## Qué es RAG y qué parte cubrimos aquí

**RAG** (*Retrieval-Augmented Generation*, generación aumentada con recuperación) resuelve un
problema concreto de los modelos de lenguaje: un LLM solo "sabe" lo que estaba en sus datos de
entrenamiento. No conoce los documentos internos de tu institución, ni lo que se publicó después
de su corte de conocimiento, ni el marco normativo específico de tu área. Y cuando no sabe, con
frecuencia **inventa** (alucina) en lugar de decir que no sabe.

La idea de RAG es no exigirle que se acuerde, sino **darle el texto relevante junto con la
pregunta**. Son tres pasos:

| Paso | Qué pasa | Herramienta |
|---|---|---|
| 1. Indexar | Convertimos cada documento del corpus en un embedding y lo guardamos | modelo de embeddings (notebooks 04 y 05) |
| 2. **Recuperar** | Convertimos la pregunta en un embedding y buscamos los documentos más cercanos | similitud coseno |
| 3. Generar | Le pasamos al LLM la pregunta **y** los documentos recuperados, y le pedimos que responda usando solo eso | LLM |

**Este notebook cubre los pasos 1 y 2**, que son los que dependen de todo lo que aprendimos hoy.
El paso 3 es el tema del siguiente bloque (`04 Modelos de IA Generativa`), donde los notebooks de
agentes sí combinan recuperación y generación.

**Ventajas de RAG:** respuestas basadas en fuentes verificables (se puede citar de dónde salió
cada dato), conocimiento actualizable sin reentrenar el modelo, y posibilidad de usar bases de
conocimiento privadas.

## Al terminar vas a poder

1. Construir un índice vectorial sobre un corpus propio y hacer búsqueda semántica por top-$k$.
2. Explicar por qué la recuperación pregunta→respuesta es un problema **asimétrico** y qué es un
   *dual encoder*.
3. Detectar el modo de falla más peligroso de RAG: recuperar algo irrelevante con aparente
   confianza, y mitigarlo con un umbral.
4. Armar el *prompt* que se le entregaría a un LLM en el paso 3.

## 0. Preparación del entorno

> **Nota técnica (la misma de los notebooks 04 y 05).** `os.environ["USE_TF"] = "0"` debe
> ejecutarse **antes** de importar `transformers` o `sentence_transformers`.

In [ ]:
# Descomenta si te falta alguna dependencia:
# !pip install sentence-transformers transformers pandas matplotlib

In [ ]:
import os
os.environ["USE_TF"] = "0"          # SIEMPRE antes de importar transformers

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore")

print("Listo.")

## 1. Paso 1 — Indexar un corpus

Vamos a construir un buscador semántico sobre un corpus real y pertinente: **16 definiciones de
los temas de este mismo módulo**. Es decir, vamos a construir un asistente que responde preguntas
sobre el curso a partir del curso.

> **Sobre el idioma.** El corpus está en inglés porque `all-MiniLM-L6-v2` solo funciona en inglés,
> como comprobamos en el notebook 04 (sección 5) y explicamos en el 05 (sección 1). Para hacer
> exactamente esto con documentos en español —que es lo que necesitarás en un proyecto real con
> datos mexicanos— basta cambiar el modelo por uno multilingüe
> (`paraphrase-multilingual-MiniLM-L12-v2`); todo el resto del código queda igual.

In [ ]:
corpus = [
    "Ordinary least squares estimates the coefficients of a linear model by minimizing the sum of squared residuals.",
    "The coefficient of determination, R squared, measures the share of the variance of the dependent variable explained by the model.",
    "Ridge regression adds a penalty proportional to the sum of the squared coefficients, which shrinks them toward zero but never sets them exactly to zero.",
    "Lasso regression adds a penalty proportional to the sum of the absolute values of the coefficients, so it can set some of them exactly to zero and perform variable selection.",
    "Principal component analysis finds new orthogonal axes that capture as much of the variance of the data as possible, and is often used to reduce dimensionality.",
    "K-means partitions observations into k groups by minimizing the distance between each observation and the centroid of its assigned cluster.",
    "The silhouette coefficient compares how close an observation is to its own cluster against the nearest other cluster, and is used to choose the number of clusters.",
    "Dynamic time warping compares two time series allowing them to be stretched or shifted in time, which the Euclidean distance cannot do.",
    "The logit model assumes the probability of a binary outcome follows a logistic function of a linear index of the covariates.",
    "An ordered logit model is used when the dependent variable has categories with a natural order, such as ratings from one to five.",
    "A confusion matrix cross-tabulates predicted classes against true classes, and from it we compute precision, recall and accuracy.",
    "A naive Bayes classifier applies Bayes theorem assuming that the features are conditionally independent given the class label.",
    "An n-gram language model estimates the probability of a word given the previous n minus one words.",
    "A random forest fits many decision trees on bootstrap samples and averages their predictions, using a random subset of predictors at each split.",
    "A word embedding maps a piece of text to a dense vector of real numbers so that texts with similar meaning end up close to each other.",
    "The transformer architecture computes attention weights between all pairs of tokens, which lets the model relate distant words in a sequence.",
]

modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# El índice vectorial: una matriz (documentos x dimensiones), con los vectores normalizados
indice = modelo.encode(corpus, normalize_embeddings=True)

print(f"Corpus indexado: {indice.shape[0]} documentos x {indice.shape[1]} dimensiones")
print(f"Memoria del índice: {indice.nbytes / 1024:.1f} KB")

**Por qué normalizamos.** En el notebook 04 (sección 3.5) comprobamos que, con vectores de
norma 1, la similitud coseno **es** el producto punto. Eso convierte toda la búsqueda en una sola
multiplicación de matrices: `indice @ vector_pregunta`. Es exactamente lo que hacen por dentro las
bases de datos vectoriales (FAISS, Pinecone, `pgvector`) para responder en milisegundos sobre
millones de documentos. Con 16 documentos podríamos hacerlo con un `for`, pero conviene escribirlo
desde el principio como se escribe en producción.

## 2. Paso 2 — Recuperar los documentos más cercanos (top-$k$)

In [ ]:
def recuperar(pregunta, k=3, umbral=None):
    """Devuelve los k documentos del corpus más cercanos a la pregunta.

    pregunta : texto de la consulta
    k        : cuántos documentos devolver
    umbral   : si se especifica, descarta los documentos por debajo de esa similitud
    """
    vector_pregunta = modelo.encode(pregunta, normalize_embeddings=True)
    similitudes = indice @ vector_pregunta                # coseno, porque todo está normalizado
    mejores = np.argsort(-similitudes)[:k]

    resultados = pd.DataFrame({
        "similitud": np.round(similitudes[mejores], 3),
        "documento": [corpus[i] for i in mejores],
    })
    if umbral is not None:
        resultados = resultados[resultados["similitud"] >= umbral]
    return resultados


recuperar("Which regression method can eliminate variables?")

In [ ]:
preguntas = [
    "Which regression method can eliminate variables?",
    "How do I decide how many groups to use?",
    "What do I do when my outcome variable is a rating from 1 to 5?",
]

for pregunta in preguntas:
    print(f"\nPREGUNTA: {pregunta}")
    for _, fila in recuperar(pregunta, k=3).iterrows():
        print(f"   {fila['similitud']:.3f}  {fila['documento']}")

**Analicemos los tres casos, incluido el que sale mal.**

- La primera y la tercera pregunta recuperan el documento correcto en primer lugar (Lasso y logit
  ordinal), **sin compartir vocabulario con él**: la pregunta dice *"eliminate variables"* y el
  documento dice *"set them exactly to zero and perform variable selection"*. Ninguna búsqueda por
  palabras clave habría encontrado eso. Ese es el valor de la búsqueda semántica.

- La segunda pregunta **falla**, y vale la pena detenerse en ella. *"How do I decide how many
  groups to use?"* debería recuperar el documento del **silhouette**, que es literalmente el que
  explica cómo elegir el número de clústeres. En su lugar, el primer lugar se lo lleva K-means
  (porque la palabra "groups" aparece ahí) y el silhouette queda en tercero, **por debajo de un
  documento sobre logit ordinal que no tiene nada que ver**.

La lección es que la recuperación es *aproximada*, y que por eso en la práctica se pide $k > 1$:
con $k = 3$ el documento correcto sí llegaría al LLM, aunque no en primer lugar. Recuperar varios
candidatos es una defensa barata contra este tipo de error.

## 3. Un problema más difícil: la recuperación es **asimétrica**

Hasta aquí comparamos textos que se parecen entre sí. Pero una pregunta y su respuesta **no se
parecen**: tienen forma gramatical distinta y muchas veces poco vocabulario en común. El caso
clásico:

In [ ]:
candidatas = [
    "What is the tallest mountain in the world?",              # la pregunta misma, repetida
    "The tallest mountain in the world is Mount Everest.",     # la respuesta correcta
    "Mount Shasta",                                            # distractor relacionado
    "I like my hike in the mountains",                         # distractor más lejano
    "I am going to a yoga class",                              # distractor sin relación
]

pregunta = "What is the tallest mountain in the world?"

vector_pregunta = modelo.encode(pregunta, normalize_embeddings=True)
vectores_candidatas = modelo.encode(candidatas, normalize_embeddings=True)
similitudes = vectores_candidatas @ vector_pregunta

print("Recuperación con un solo modelo de embeddings (similitud 'pura'):\n")
for i in np.argsort(-similitudes):
    print(f"   {similitudes[i]:.3f}   {candidatas[i]}")

print(f"\nMejor candidata según el modelo: '{candidatas[np.argmax(similitudes)]}'")

**Mira cuál ganó.** El sistema devuelve como "mejor respuesta" **la pregunta misma**, con
similitud 1.000, porque es idéntica carácter por carácter. La respuesta correcta —la que menciona
el Everest— queda en segundo lugar.

No es un detalle de laboratorio: es el modo de falla natural de este enfoque. `all-MiniLM-L6-v2`
fue entrenado para **similitud simétrica** (¿la oración A se parece a la oración B?), y bajo ese
criterio la respuesta perfecta a una pregunta es repetir la pregunta. Pero la tarea real es
**asimétrica**: queremos el pasaje que *contesta*, no el que *se parece*.

### 3.1 Dual encoders: DPR

**DPR** (*Dense Passage Retrieval*, Karpukhin et al., 2020) resuelve esto con **dos codificadores
distintos**, entrenados juntos pero especializados:

- un `question_encoder`, que codifica preguntas, y
- un `context_encoder`, que codifica los pasajes candidatos.

Se entrenan de modo que el vector de una pregunta quede cerca del vector de su respuesta correcta,
**aunque no compartan palabras**. Como la pregunta y el pasaje pasan por modelos *distintos*, una
pregunta ya no puede tener similitud 1.000 consigo misma: sus dos representaciones ni siquiera
vienen del mismo modelo.

> ⚠️ **Advertencia de descarga: ~846 MB** (dos modelos de ~423 MB). En la primera ejecución puede
> tardar varios minutos. **Si están en clase con la red compartida, no la corran todos al mismo
> tiempo**; basta con que el profesor la corra y discutamos el resultado. Cambia
> `EJECUTAR_DPR = False` para saltarte esta sección.

In [ ]:
EJECUTAR_DPR = True          # ponlo en False para saltarte la descarga de ~846 MB

if EJECUTAR_DPR:
    import torch
    from transformers import AutoTokenizer, DPRContextEncoder, DPRQuestionEncoder

    tok_contexto = AutoTokenizer.from_pretrained("facebook/dpr-ctx_encoder-multiset-base")
    enc_contexto = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-multiset-base")

    tok_pregunta = AutoTokenizer.from_pretrained("facebook/dpr-question_encoder-multiset-base")
    enc_pregunta = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-multiset-base")

    print("Modelos DPR cargados.")
    print(f"Dimensión de salida: {enc_pregunta.config.hidden_size}")

In [ ]:
if EJECUTAR_DPR:

    def vector_dpr(texto, tokenizador, codificador):
        entradas = tokenizador(texto, return_tensors="pt", truncation=True, max_length=512)["input_ids"]
        with torch.no_grad():
            salida = codificador(entradas).pooler_output
        return salida.flatten().numpy()

    def coseno(u, v):
        return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

    v_pregunta_dpr = vector_dpr(pregunta, tok_pregunta, enc_pregunta)
    sim_dpr = np.array([
        coseno(v_pregunta_dpr, vector_dpr(c, tok_contexto, enc_contexto))
        for c in candidatas
    ])

    comparacion = pd.DataFrame({
        "candidata": candidatas,
        "similitud 'pura'": np.round(similitudes, 3),
        "dual encoder (DPR)": np.round(sim_dpr, 3),
    })
    comparacion["lugar 'pura'"] = comparacion["similitud 'pura'"].rank(ascending=False).astype(int)
    comparacion["lugar DPR"] = comparacion["dual encoder (DPR)"].rank(ascending=False).astype(int)

    print(f"Mejor candidata con similitud 'pura': {candidatas[np.argmax(similitudes)]}")
    print(f"Mejor candidata con DPR            : {candidatas[np.argmax(sim_dpr)]}\n")

    display(comparacion)

**Compara las dos columnas de "lugar".** DPR coloca en primer lugar la **respuesta**
(*"The tallest mountain in the world is Mount Everest."*) y relega la pregunta repetida al segundo,
que es el comportamiento que queremos en un sistema de preguntas y respuestas.

Los valores absolutos de similitud de DPR son más bajos y más comprimidos que los del modelo
simétrico: **no son comparables entre modelos**. Lo único comparable —y lo único que importa para
recuperar— es el **orden**.

**Cuál usar en la práctica.** No siempre necesitas un dual encoder: si tus consultas se parecen a
tus documentos (buscar noticias parecidas, agrupar respuestas de encuesta, detectar duplicados),
un modelo simétrico basta y es 10 veces más ligero. El dual encoder se justifica cuando la consulta
y el documento son objetos gramaticalmente distintos: preguntas contra pasajes, o consultas cortas
contra documentos largos.

## 4. El modo de falla que hay que conocer: la búsqueda **siempre** devuelve algo

Este es el punto más importante del notebook desde el punto de vista de uso responsable.

`np.argsort` siempre regresa un primer lugar. **Aunque la respuesta no exista en el corpus.** Si
conectas eso directamente a un LLM, el modelo recibirá contexto irrelevante presentado como si
fuera pertinente, y con frecuencia construirá encima de él una respuesta que suena bien y es falsa.
Es decir: RAG **reduce** las alucinaciones, pero mal implementado también puede **fabricarlas con
apariencia de fuente**.

Probemos con dos preguntas que nuestro corpus del curso no puede contestar.

In [ ]:
preguntas_fuera = [
    "What is the best recipe for tacos al pastor?",
    "Who won the 2022 football world cup?",
]

print("PREGUNTAS QUE EL CORPUS *NO* PUEDE CONTESTAR (sin umbral):\n")
for pregunta_fuera in preguntas_fuera:
    print(f"PREGUNTA: {pregunta_fuera}")
    for _, fila in recuperar(pregunta_fuera, k=2).iterrows():
        print(f"   {fila['similitud']:.3f}  {fila['documento'][:95]}...")
    print()

In [ ]:
# Comparemos el mejor puntaje de las preguntas que SÍ pertenecen al corpus
# contra el de las que NO pertenecen.
filas = []
for pregunta_i in preguntas:
    filas.append({"pregunta": pregunta_i[:52], "¿en el corpus?": "sí",
                  "mejor similitud": recuperar(pregunta_i, k=1)["similitud"].iloc[0]})
for pregunta_i in preguntas_fuera:
    filas.append({"pregunta": pregunta_i[:52], "¿en el corpus?": "NO",
                  "mejor similitud": recuperar(pregunta_i, k=1)["similitud"].iloc[0]})

diagnostico = pd.DataFrame(filas)
print(diagnostico.to_string(index=False))

UMBRAL = 0.20
print(f"\nCon un umbral de {UMBRAL}:")
for pregunta_i in preguntas + preguntas_fuera:
    encontrados = recuperar(pregunta_i, k=3, umbral=UMBRAL)
    if len(encontrados) == 0:
        print(f"   [SIN RESULTADOS] {pregunta_i}")
    else:
        print(f"   [{len(encontrados)} documento(s)] {pregunta_i}")

**Los números separan bien los dos grupos:** las preguntas contestables por el corpus
alcanzan similitudes claramente más altas que las que no lo son. Un umbral intermedio permite que
el sistema responda *"no tengo información sobre eso"* en vez de entregar un documento al azar.

**Tres advertencias antes de que te lleves el 0.20 a tu proyecto:**

1. **El umbral cuesta.** Míralo en la salida: con 0.20, la segunda pregunta pasa de 3 documentos
   a 2, y el que se cae es justamente el del silhouette (0.179), que era **el correcto**. Subir el
   umbral protege contra el ruido y al mismo tiempo tira respuestas buenas. No hay un valor
   gratuito.
2. **El umbral no es universal.** Depende del modelo, del corpus y del tipo de pregunta. Hay que
   **calibrarlo con datos propios**: junta preguntas que sí deberían tener respuesta y preguntas
   que no, y elige el corte que mejor las separe. Es el mismo ejercicio de la Sesión 5 al elegir
   el punto de corte de una regresión logística, con la misma disyuntiva entre falsos positivos y
   falsos negativos.
3. **Separación no es garantía.** Que estas cinco preguntas se separen limpiamente no significa
   que todas lo hagan; una pregunta fuera del corpus pero temáticamente cercana puede colarse por
   encima del umbral.
4. **El umbral no mide veracidad.** Solo mide parecido. Un documento del corpus que sea muy
   parecido a la pregunta pero esté desactualizado o equivocado pasará el umbral sin problema.

## 5. Paso 3 (adelanto): armar el *prompt*

Ya tenemos los documentos. El último paso de RAG es entregárselos al LLM junto con la pregunta y
con instrucciones explícitas sobre cómo usarlos. Aquí **no** vamos a llamar a ningún modelo
—eso es el siguiente bloque—, pero sí vale la pena ver el texto exacto que se le enviaría, porque
ahí es donde se juega buena parte de la calidad del sistema.

In [ ]:
def armar_prompt(pregunta, k=3, umbral=0.20):
    recuperados = recuperar(pregunta, k=k, umbral=umbral)

    if len(recuperados) == 0:
        return ("[No se recuperó ningún documento por encima del umbral. "
                "El sistema debería responder que no tiene información suficiente, "
                "sin llamar al LLM.]")

    contexto = "\n".join(
        f"[{i + 1}] {fila['documento']}" for i, (_, fila) in enumerate(recuperados.iterrows())
    )

    return f"""Eres un asistente del Módulo V del Diplomado de Ciencia de Datos.
Responde la pregunta del usuario usando ÚNICAMENTE los fragmentos de contexto que aparecen abajo.
Cita entre corchetes el número del fragmento en el que te basas.
Si el contexto no contiene la respuesta, di explícitamente que no cuentas con esa información;
no la inventes ni la completes con conocimiento propio.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""


print(armar_prompt("Which regression method can eliminate variables?"))

In [ ]:
print(armar_prompt("What is the best recipe for tacos al pastor?"))

**Fíjate en las tres instrucciones del *prompt*,** porque cada una responde a un riesgo
concreto que ya vimos:

1. *"usando ÚNICAMENTE los fragmentos de contexto"* — evita que el LLM mezcle lo que recuperamos
   con lo que "creía saber", que es justo lo que RAG intenta impedir.
2. *"Cita entre corchetes el número del fragmento"* — hace la respuesta **auditable**: quien la
   lea puede ir al documento original y verificarla. En un trabajo académico o de política pública
   esto no es opcional.
3. *"Si el contexto no contiene la respuesta, di explícitamente que no cuentas con esa
   información"* — es la segunda línea de defensa, después del umbral, contra la alucinación.

Y observa el segundo caso: cuando el umbral filtra todo, **ni siquiera hay que llamar al LLM**. El
sistema puede responder "no tengo esa información" sin gastar una sola llamada. La calidad de un
sistema RAG se decide tanto en la recuperación como en la generación.

## Para pensar

1. En la sección 2, la pregunta *"How do I decide how many groups to use?"* dejó el documento
   correcto (silhouette) en tercer lugar, debajo de uno irrelevante. Con $k = 3$ el error queda
   compensado. Explica qué pasaría con $k = 1$, y cuál es el costo de subir $k$ a 10 (piensa en
   la ventana de contexto y el costo por token del LLM que discutimos en las notas).

2. En la sección 3, la similitud "pura" devolvió **la pregunta misma** como mejor respuesta.
   Además de cambiar a un dual encoder, se te ocurre una solución mucho más barata para este caso
   particular. ¿Cuál es, y en qué situación real dejaría de funcionar?

3. Los puntajes de similitud de DPR y los del modelo simétrico están en escalas distintas. ¿Por
   qué sería un error tomar el umbral de 0.20 que calibramos en la sección 4 y aplicarlo tal cual
   a los puntajes de DPR?

4. RAG se vende como la solución a las alucinaciones. Con lo que viste en la sección 4, escribe en
   tres o cuatro líneas una respuesta matizada a la afirmación *"con RAG el modelo ya no inventa"*.
   Menciona al menos dos formas en que un sistema RAG puede producir una respuesta falsa **con
   fuente citada**.

5. **Ejercicio.** Sustituye el corpus por 15 o 20 fragmentos de un documento de tu área (un
   reglamento, un informe, las notas de este módulo) y prueba cinco preguntas: tres que el
   documento sí conteste y dos que no. Calibra el umbral que mejor las separe y reporta los
   errores que comete.

6. **Ejercicio.** Los documentos de nuestro corpus son de una sola oración. En un caso real
   tendrías archivos de decenas de páginas y habría que partirlos en fragmentos (*chunking*).
   Discute qué pasa si los fragmentos son demasiado largos (¿qué le ocurre al embedding de un
   fragmento que habla de cinco cosas distintas?) y qué pasa si son demasiado cortos (¿y si la
   respuesta queda partida entre dos fragmentos?).

---

**Cierre del bloque.** Con esto termina el bloque de embeddings. En `04 Modelos de IA Generativa`
retomamos justo donde nos quedamos: el paso 3 de RAG, es decir, entregarle este *prompt* a un LLM
y, más allá, darle herramientas para que actúe (agentes).